# Réseau de neurones simple avec TensorFlow

Dans ce TP, nous allons construire deux réseaux de neurones avec l'API de base de TensorFlow (`tf.Variable`, `tf.Module`, `tf.GradientTape`, `tf.data`), sans modèle Keras : un réseau sans couche cachée d'abord, puis un perceptron multicouche (PMC). Le TP suivant construira les mêmes réseaux avec Keras.

Les données sont celles de MNIST, le jeu *Hello World!* du machine learning : des images de 28 $\times$ 28 pixels en niveaux de gris, qui représentent des chiffres de 0 à 9. Chaque image est associée à un label, le chiffre que l'image est censée représenter.

Les étapes :

1. Prendre en main les tenseurs TensorFlow
2. Explorer et préparer les données, mettre de côté un ensemble de validation
3. Écrire un réseau sans couche cachée, sa perte, sa justesse et son pipeline `tf.data`
4. L'entraîner par descente de gradient écrite à la main, puis avec un optimiseur Keras
5. Compiler le pas d'entraînement avec `tf.function`
6. Passer à un PMC, le régulariser, choisir ses hyper-paramètres sur la validation
7. L'évaluer sur l'ensemble de test, le sauvegarder et le restaurer

Ce notebook n'a pas besoin de GPU : chaque entraînement dure de quelques dizaines de secondes à quelques minutes sur le processeur de Colab.

## Imports

In [ ]:
import random
import time
from collections.abc import Callable

import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn
import tensorflow as tf

TensorFlow fournit les tenseurs, les gradients et le pipeline de données ; Keras, son jeu MNIST, sa fonction de perte et ses optimiseurs. [`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) et [`seaborn`](https://seaborn.pydata.org/) servent aux graphiques et aux opérations simples sur les tableaux.

## Prise en main des tenseurs

Sur Colab, TensorFlow est déjà installé. Sur votre machine, installez-le avec `pip install tensorflow`.

Avant de construire des réseaux, manipulons les tenseurs TensorFlow, dont l'API est proche de celle de NumPy (voir les slides « Tenseurs : comme des tableaux NumPy » et « Tenseurs : deux différences avec NumPy »). Les opérations qui échouent lèvent une `tf.errors.InvalidArgumentError` : attrapez-la avec `try`/`except` pour afficher son message.

1. Créez un tenseur `a` de forme (2, 3) contenant les entiers de 0 à 5 (`tf.range`, puis `tf.reshape`). Quel est son `dtype` ? Convertissez-le en `float32` avec `tf.cast`, puis créez un tenseur `b` de forme (3,) rempli de 1 (`tf.ones`).
2. Calculez `a + b` et le produit matriciel `a @ tf.ones((3, 2))`. Quelles sont les formes des résultats ? Pourquoi `a + tf.ones(2)` échoue-t-il ? Et `a @ b` ?
3. Extrayez la deuxième colonne de `a`, puis sa dernière ligne. Ajoutez une dimension de taille 1 en tête de `a` avec `tf.expand_dims` : quelle forme obtient-on ? À quoi sert une telle dimension avec un modèle ?
4. Convertissez `a` en tableau NumPy (`.numpy()`), puis de nouveau en tenseur (`tf.convert_to_tensor`). Convertissez de même `np.ones(3)` : quel `dtype` obtient-on, et que se passe-t-il si on l'ajoute à `a` ? Enfin, listez les GPU que voit TensorFlow (`tf.config.list_physical_devices`).

**Documentation utile**

- [`tf.range`](https://www.tensorflow.org/api_docs/python/tf/range)
- [`tf.reshape`](https://www.tensorflow.org/api_docs/python/tf/reshape)
- [`tf.ones`](https://www.tensorflow.org/api_docs/python/tf/ones)
- [`tf.expand_dims`](https://www.tensorflow.org/api_docs/python/tf/expand_dims)
- [`tf.config.list_physical_devices`](https://www.tensorflow.org/api_docs/python/tf/config/list_physical_devices)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tf.reshape(tf.range(6), (2, 3))`, puis `tf.cast(a, tf.float32)`
- `a[:, 1]` : la deuxième colonne ; `a[-1]` : la dernière ligne
- `tf.expand_dims(a, axis=0)` : une dimension de taille 1 en position 0
- `tf.config.list_physical_devices("GPU")` : la liste des GPU visibles, vide sans GPU
- Une opération qui échoue, dans un `try` :

```python
try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as error:
  print(error)
```

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
a = tf.reshape(..., (2, 3))
print(a.dtype)
a = tf.cast(a, ...)
b = ...

print((a + b).shape, (a @ tf.ones((3, 2))).shape)
try:
  ...  # a + tf.ones(2), puis a @ b dans un second try
except tf.errors.InvalidArgumentError as error:
  print(error)

print(a[:, 1], ...)  # deuxième colonne, dernière ligne
print(tf.expand_dims(a, axis=0).shape)

print(tf.convert_to_tensor(a.numpy()).dtype, tf.convert_to_tensor(np.ones(3)).dtype)
try:
  ...  # a + le tenseur créé depuis np.ones(3)
except tf.errors.InvalidArgumentError as error:
  print(error)
print(tf.config.list_physical_devices(...))
```

### Solution

In [ ]:
a = tf.reshape(tf.range(6), (2, 3))
print(a, a.dtype)
a = tf.cast(a, tf.float32)
b = tf.ones(3)

# b est diffusé (broadcasting) sur chaque ligne de a
print(a + b, (a + b).shape)
# Produit matriciel : (2, 3) @ (3, 2) -> (2, 2)
print(a @ tf.ones((3, 2)))

try:
  a + tf.ones(2)
except tf.errors.InvalidArgumentError as error:
  print("a + tf.ones(2) :", error)
try:
  a @ b
except tf.errors.InvalidArgumentError as error:
  print("a @ b :", error)

print(a[:, 1])  # deuxième colonne
print(a[-1])  # dernière ligne
print(tf.expand_dims(a, axis=0).shape)  # dimension de batch de taille 1

print(type(a.numpy()), tf.convert_to_tensor(a.numpy()).dtype)
ones_64 = tf.convert_to_tensor(np.ones(3))
print(ones_64.dtype)
try:
  a + ones_64
except tf.errors.InvalidArgumentError as error:
  print("a + ones_64 :", error)
print(tf.config.list_physical_devices("GPU"))

- `tf.range` crée des entiers 32 bits (`int32`) ; les réseaux de neurones calculent en flottants, par défaut `float32`.
- `a + b` a la forme (2, 3) et `a @ tf.ones((3, 2))` la forme (2, 2). Pour le broadcasting, les dimensions sont comparées en partant de la dernière : elles doivent être égales, ou l'une d'elles doit valoir 1 ou être absente. (2, 3) et (3,) sont compatibles, mais pas (2, 3) et (2,) : il faudrait la forme (2, 1), `tf.ones((2, 1))`.
- `a @ b` échoue aussi : en TensorFlow, `@` multiplie des matrices (au moins deux dimensions), alors que NumPy accepte un vecteur. Il faut `b` de forme (3, 1), ou `tf.linalg.matvec(a, b)`.
- `tf.expand_dims(a, axis=0)` a la forme (1, 2, 3). Les modèles traitent des batchs, la première dimension étant celle des exemples : pour prédire sur un seul exemple, on lui ajoute une dimension de batch de taille 1.
- `tf.convert_to_tensor` conserve le type NumPy : `float32` pour `a.numpy()`, mais `float64` pour `np.ones(3)`, le type par défaut de NumPy. TensorFlow ne convertit jamais un type en un autre de lui-même : `float32 + float64` lève une erreur, là où NumPy promeut en `float64`. D'où le `tf.cast`, ou `.astype("float32")` côté NumPy, que l'on fera sur les images de MNIST.
- Sans GPU, la liste est vide et TensorFlow calcule sur le processeur ; avec un GPU, il l'utilise d'office.

## Récupération des données

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Regardons les données

Utilisez la cellule suivante pour explorer les tableaux NumPy `X_train`, `y_train`, `X_test` et `y_test` : leur forme (`shape`), leur type (`dtype`), leurs valeurs extrêmes, et quelques images avec `plt.imshow`.

Vous essayerez de répondre aux questions suivantes :

- Combien y a-t-il d'images au total dans ce dataset ?
- Sous quelle forme sont stockées les images ? les labels ?
- Les classes sont-elles équilibrées ?

**Documentation utile**

- [`numpy.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()`, `X_train.max()` : forme, type et valeurs extrêmes
- `np.unique(y_train, return_counts=True)` : les classes et le nombre d'exemples de chacune
- `plt.imshow(X_train[0], cmap="gray_r")` : la première image, en noir sur blanc

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, ...)
print(X_test.shape, ...)

classes, counts = np.unique(..., return_counts=True)
print(classes, counts)

plt.imshow(..., cmap="gray_r")
plt.title(f"Label : {y_train[0]}")
plt.show()
```

### Solution

In [ ]:
print(f"Images d'entraînement : {X_train.shape}, {X_train.dtype}")
print(f"Valeurs des pixels : de {X_train.min()} à {X_train.max()}")
print(f"Labels d'entraînement : {y_train.shape}, {y_train.dtype}")
print(f"Images et labels de test : {X_test.shape}, {y_test.shape}")
classes, counts = np.unique(y_train, return_counts=True)
print(f"Exemples par classe : {dict(zip(classes.tolist(), counts.tolist()))}")

plt.imshow(X_train[0], cmap="gray_r")
plt.title(f"Premier exemple du dataset ({y_train[0]})")
plt.show()

seaborn.countplot(x=y_train)
plt.title("Décompte des différentes classes (chiffres)")
plt.ylabel("Décompte")
plt.xlabel("Chiffre")
plt.show()

- 70 000 images : 60 000 d'entraînement et 10 000 de test.
- Les images sont un tableau `uint8` de forme (60 000, 28, 28) : un entier de 0 (fond) à 255 (encre) par pixel. Les labels sont un tableau `uint8` de forme (60 000,), un chiffre de 0 à 9 par image.
- Les classes sont à peu près équilibrées : de 5 421 exemples (le 5) à 6 742 (le 1). La justesse, la proportion de bonnes réponses, est donc une métrique pertinente.

## Affichage d'exemples

Affichez 25 exemples, tirés au hasard dans la base d'entraînement, sur une grille de 5 $\times$ 5. Affichez le label de chacun dans son titre.

**Documentation utile**

- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Une grille de 5 x 5 sous-graphiques
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.random.choice(len(X_train), size=(5, 5), replace=False)` : 25 indices distincts, rangés en grille
- `ax[i, j].imshow(image, cmap="gray_r")`, `ax[i, j].set_title(...)` et `ax[i, j].axis("off")` : l'image, son titre et sans axes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
f, ax = plt.subplots(5, 5, figsize=(15, 15))
indices = np.random.choice(..., size=(5, 5), replace=False)
for i in range(5):
  for j in range(5):
    index = indices[i, j]
    ax[i, j].imshow(..., cmap="gray_r")
    ax[i, j].set_title(f"Exemple {index} ({...})")
    ax[i, j].axis("off")
```

### Solution

In [ ]:
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# 25 indices au hasard, sans remise (on ne veut pas afficher la même image deux
# fois)
indices = np.random.choice(len(X_train), size=(5, 5), replace=False)

for i in range(5):
  for j in range(5):
    index = indices[i, j]
    ax[i, j].imshow(X_train[index], cmap="gray_r")
    ax[i, j].set_title(f"Exemple {index} ({y_train[index]})")
    ax[i, j].axis("off")

## Transformation des données

Trois transformations préparent les images pour le réseau :

1. Les mettre à plat : un réseau sans couche cachée prend un vecteur par exemple, de forme `(_, 28²)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt qu'entre $0$ et $255$.
3. Les convertir en `float32` : la division de NumPy donne des `float64`, que TensorFlow refuse de mélanger aux `float32` de ses variables.

*Appliquez ces trois transformations à `X_train` et à `X_test`.*

**Documentation utile**

- [`numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html)
- [`numpy.ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X.reshape(-1, 28 * 28)` : `-1` laisse NumPy calculer le nombre de lignes
- `X / 255` : des valeurs entre 0 et 1, en `float64`
- `.astype("float32")` : la conversion en `float32`

Attendu : `X_train.shape` vaut `(60000, 784)`, `X_train.dtype` vaut `float32` et `X_train.max()` vaut 1.0.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = (X_train.reshape(-1, ...) / ...).astype(...)
X_test = ...
print(X_train.shape, X_train.dtype, X_train.max())
```

### Solution

In [ ]:
input_dim = 28 * 28

# Mise à plat : un vecteur de input_dim pixels par image, puis passage dans
# [0, 1] et conversion en float32
X_train = (X_train.reshape(-1, input_dim) / 255).astype("float32")
X_test = (X_test.reshape(-1, input_dim) / 255).astype("float32")
print(X_train.shape, X_train.dtype, X_train.max())

## Séparation entraînement / validation

Pour choisir les hyper-paramètres (learning rate, taille du réseau…), on compare des modèles sur un ensemble de **validation**. L'ensemble de test est réservé à l'évaluation finale : s'en servir pour faire des choix rendrait cette évaluation trop optimiste.

On met de côté les 10 000 derniers exemples d'entraînement pour la validation :

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Réseau de neurones sans couche cachée

Un réseau de neurones sans couche cachée calcule $\text{softmax}(XW + b)$ : c'est la **régression logistique multinomiale**. `W` est la matrice des poids, multipliée à l'entrée, et `b` le vecteur des biais, ajouté ensuite. La softmax transforme les 10 scores de chaque exemple, ses *logits*, en probabilités.

### Initialisation des poids

*Complétez la fonction `init_weights(n_in, n_out)`, qui renvoie une `tf.Variable` de forme `(n_in, n_out)` tirée d'une loi normale (`tf.random.normal`), divisée par $\sqrt{n_\text{in}}$ (slide « PMC en TensorFlow de base : le modèle »).*

Cette division donne aux poids une variance de $1 / n_\text{in}$ : la sortie de la couche, une somme de $n_\text{in}$ termes, reste de l'ordre de grandeur de ses entrées, quelle que soit leur nombre.

**Documentation utile**

- [`tf.random.normal`](https://www.tensorflow.org/api_docs/python/tf/random/normal)

In [ ]:
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  pass  # Votre code ici

#### Conseils supplémentaires

- `tf.random.normal((n_in, n_out))` : un tenseur de forme `(n_in, n_out)`, de moyenne 0 et d'écart type 1
- `n_in**0.5` : la racine carrée de `n_in`
- `tf.Variable(tenseur)` : une variable, que TensorFlow pourra mettre à jour

Attendu : `init_weights(784, 10)` a la forme (784, 10) et un écart type proche de $1 / 28 \approx 0{,}036$ (`tf.math.reduce_std`).

```python
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  return tf.Variable(tf.random.normal(...) / ...)
```

#### Solution

In [ ]:
def init_weights(n_in: int, n_out: int) -> tf.Variable:
  # Variance 1 / n_in : des sorties de l'ordre de 1, quel que soit n_in
  return tf.Variable(tf.random.normal((n_in, n_out)) / n_in**0.5)


W = init_weights(784, 10)
print(W.shape, float(tf.math.reduce_std(W)))

### Définition du modèle

*Complétez la classe `SimpleNN` : dans `__init__`, créez `self.W` avec `init_weights` et `self.b`, une `tf.Variable` de zéros de forme `(n_outputs,)` ; `__call__` renvoie les logits $XW + b$.*

Le modèle ne calcule pas la softmax : la fonction de perte l'appliquera elle-même (`from_logits=True`), de façon plus stable numériquement (slide « PMC en TensorFlow de base : un pas d'entraînement »).

In [ ]:
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    # Votre code ici

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Votre code ici

#### Conseils supplémentaires

- `tf.Variable(tf.zeros(n_outputs))` : les biais, nuls au départ
- `X @ self.W + self.b` : le produit matriciel, puis l'ajout des biais, diffusé sur chaque ligne

Attendu : `SimpleNN()(X_train[:5])` a la forme (5, 10), et `SimpleNN().trainable_variables` contient 2 variables.

```python
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W = init_weights(..., ...)
    self.b = tf.Variable(...)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    return ...
```

#### Solution

In [ ]:
class SimpleNN(tf.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W = init_weights(n_inputs, n_outputs)
    self.b = tf.Variable(tf.zeros(n_outputs))

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Les logits : la softmax est appliquée par la fonction de perte
    return X @ self.W + self.b


simple_nn = SimpleNN()
print(simple_nn(X_train[:5]).shape, len(simple_nn.trainable_variables))

### Calcul de la fonction de perte

La fonction de perte standard en classification multi-classes est l'entropie croisée. Keras la fournit pour des labels entiers (0 à 9, plutôt que des vecteurs one-hot) : `keras.losses.sparse_categorical_crossentropy`, avec `from_logits=True` puisque le modèle renvoie des logits. Elle rend une valeur par exemple : pour pouvoir l'interpréter directement, on en prend la moyenne avec `tf.reduce_mean`.

*Complétez la fonction `categorical_crossentropy`. Quelle valeur prend-elle pour un modèle qui n'a pas encore appris ?*

**Documentation utile**

- [`keras.losses.sparse_categorical_crossentropy`](https://keras.io/api/losses/probabilistic_losses/#sparse_categorical_crossentropy-function)

In [ ]:
def categorical_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  pass  # Votre code ici

#### Conseils supplémentaires

- `keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)` : les labels d'abord, puis les logits ; une valeur par exemple
- `tf.reduce_mean(...)` : leur moyenne, un scalaire

Un modèle qui n'a pas appris donne à peu près la même probabilité, 1/10, à chaque classe : sa perte vaut environ $-\log(1/10) = \log 10 \approx 2{,}3$.

```python
def categorical_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  cross_entropy = keras.losses.sparse_categorical_crossentropy(
    ..., ..., from_logits=...
  )
  return tf.reduce_mean(...)


print(categorical_crossentropy(y_train, SimpleNN()(X_train)))
```

#### Solution

In [ ]:
def categorical_crossentropy(y: np.ndarray, logits: tf.Tensor) -> tf.Tensor:
  # from_logits=True : la softmax est calculée dans la perte, de façon stable
  cross_entropy = keras.losses.sparse_categorical_crossentropy(
    y, logits, from_logits=True
  )
  return tf.reduce_mean(cross_entropy)


# Environ log(10) = 2,3 : le modèle répond presque au hasard
print(categorical_crossentropy(y_train, SimpleNN()(X_train)))

### Métriques

Pour savoir si un modèle apprend, on mesure ses performances. Nous utiliserons la plus simple, mais aussi une des plus informatives : la justesse (*accuracy*), le nombre de bonnes prédictions sur le nombre total de prédictions.

*Complétez la fonction `accuracy`. La classe prédite est celle du plus grand logit (`tf.argmax`) : comparez-la aux labels, puis calculez la proportion de bonnes réponses.*

In [ ]:
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  pass  # Votre code ici

#### Conseils supplémentaires

- `tf.argmax(logits, axis=1)` : la classe prédite pour chaque exemple (la softmax ne change pas l'ordre des scores, inutile de la calculer)
- `predictions == y` : un tenseur de booléens
- `tf.cast(..., tf.float32)` puis `tf.reduce_mean` : la proportion de `True`
- `float(...)` : un nombre Python, plus simple à afficher et à stocker

Attendu : environ 0,1 pour un modèle qui n'a pas appris, une chance sur dix.

```python
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(..., axis=...)
  return float(tf.reduce_mean(tf.cast(..., tf.float32)))


print(accuracy(y_val, SimpleNN()(X_val)))
```

#### Solution

In [ ]:
def accuracy(y: np.ndarray, logits: tf.Tensor) -> float:
  predictions = tf.argmax(logits, axis=1)
  return float(tf.reduce_mean(tf.cast(predictions == y, tf.float32)))


print(accuracy(y_val, SimpleNN()(X_val)))

## Création d'un dataset TensorFlow

Le modèle apprendra par mini-batchs : à chaque pas, un petit groupe d'exemples tirés au hasard. `tf.data` les prépare (slides « `tf.data` : fournir les données » et « `tf.data` sur MNIST »).

*Complétez la fonction `create_dataset`, qui crée un `tf.data.Dataset` à partir des tableaux `X` et `y` avec `tf.data.Dataset.from_tensor_slices`, le mélange avec `shuffle` à chaque epoch, le découpe en batchs `(X_batch, y_batch)` de `batch_size` exemples avec `batch`, et prépare le batch suivant pendant le calcul avec `prefetch`.*

In [ ]:
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  pass  # Votre code ici

### Conseils supplémentaires

- `tf.data.Dataset.from_tensor_slices((X, y))` : un dataset d'exemples `(x, y)`
- `.shuffle(len(X))` **avant** `.batch(batch_size)` : on mélange les exemples ; après, on ne mélangerait que l'ordre des mêmes batchs. Le tampon de `len(X)` exemples donne un mélange complet, refait à chaque epoch.
- `.prefetch(tf.data.AUTOTUNE)` : en dernier

Attendu : `create_dataset(X_train, y_train)` compte 391 batchs (`len`), et le premier, `next(iter(...))`, a des images de forme (128, 784).

```python
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices(...)
  dataset = dataset.shuffle(...)
  dataset = dataset.batch(...)
  return dataset.prefetch(...)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)
```

### Solution

In [ ]:
def create_dataset(
  X: np.ndarray, y: np.ndarray, batch_size: int = 128
) -> tf.data.Dataset:
  dataset = tf.data.Dataset.from_tensor_slices((X, y))
  # Mélange des exemples, avant le découpage en batchs, refait à chaque epoch
  dataset = dataset.shuffle(len(X))
  dataset = dataset.batch(batch_size)
  # Préparation du batch suivant pendant le calcul sur le batch courant
  return dataset.prefetch(tf.data.AUTOTUNE)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)

## Apprentissage

Voici une boucle d'apprentissage presque complète : elle parcourt les batchs à chaque epoch, puis mesure la perte et la justesse sur les ensembles d'entraînement et de validation, et trace leurs courbes à la fin.

*Implémentez la partie manquante : la mise à jour des variables du modèle pour un batch, par descente de gradient. Calculez la perte sous un `tf.GradientTape`, en lui ajoutant `regularization(model)` quand une régularisation est donnée (elle servira plus loin), puis ses gradients, et retirez de chaque variable `learning_rate` fois son gradient avec `assign_sub` (slide « Régression linéaire par descente de gradient »).*

In [ ]:
LossFunction = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = Callable[[tf.Module], tf.Tensor]


def train(
  model: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  y_train_pred = model(X_train)
  y_val_pred = model(X_val)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [float(loss(y_train, y_train_pred))]
  val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Métriques initiales : acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Votre code ici

    # Calcul et affichage des métriques toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      y_train_pred = model(X_train)
      y_val_pred = model(X_val)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(float(loss(y_train, y_train_pred)))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {e + 1} : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Conseils supplémentaires

- `with tf.GradientTape() as tape:` : les opérations du bloc sont enregistrées
- `loss(y_batch, model(X_batch))` : la perte du batch ; `+= regularization(model)` si `regularization is not None`
- `tape.gradient(loss_value, model.trainable_variables)` : la liste des gradients, dans l'ordre des variables
- `variable.assign_sub(learning_rate * gradient)` : la mise à jour, pour chaque couple de `zip(gradients, model.trainable_variables)`

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_batch, y_batch in dataset_train:
  with tf.GradientTape() as tape:
    loss_value = ...
    if regularization is not None:
      loss_value += ...
  gradients = tape.gradient(..., model.trainable_variables)
  for gradient, variable in zip(gradients, model.trainable_variables):
    variable.assign_sub(...)
```

### Solution

In [ ]:
LossFunction = Callable[[np.ndarray, tf.Tensor], tf.Tensor]
RegularizationFunction = Callable[[tf.Module], tf.Tensor]


def train(
  model: tf.Module,
  X_train: np.ndarray = X_train,
  y_train: np.ndarray = y_train,
  X_val: np.ndarray = X_val,
  y_val: np.ndarray = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  y_train_pred = model(X_train)
  y_val_pred = model(X_val)
  accuracies = [accuracy(y_train, y_train_pred)]
  val_accuracies = [accuracy(y_val, y_val_pred)]
  losses = [float(loss(y_train, y_train_pred))]
  val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Métriques initiales : acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      # Calcul de la perte, opérations enregistrées par la tape
      with tf.GradientTape() as tape:
        loss_value = loss(y_batch, model(X_batch))
        if regularization is not None:
          loss_value += regularization(model)

      # Gradient de la perte par rapport à chaque variable du modèle
      gradients = tape.gradient(loss_value, model.trainable_variables)

      # Mise à jour de chaque variable selon la règle :
      # nouvelle_valeur = ancienne_valeur - learning_rate * gradient
      for gradient, variable in zip(gradients, model.trainable_variables):
        variable.assign_sub(learning_rate * gradient)

    # Calcul et affichage des métriques toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      y_train_pred = model(X_train)
      y_val_pred = model(X_val)
      accuracies.append(accuracy(y_train, y_train_pred))
      losses.append(float(loss(y_train, y_train_pred)))
      val_accuracies.append(accuracy(y_val, y_val_pred))
      val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {e + 1} : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Entraînement du réseau sans couche cachée

10 epochs, avec un learning rate de 0,5 :

In [ ]:
simple_nn = SimpleNN()
_ = train(simple_nn, epochs=10, learning_rate=0.5)

*Quelle justesse de validation atteint ce modèle ? Progresse-t-il encore après quelques epochs ?*

Votre réponse ici

#### Solution

Environ 92,5 % de justesse en validation. L'essentiel est appris dès la première epoch (environ 91 %) ; ensuite, le modèle ne gagne plus qu'un point en 9 epochs. Les justesses d'entraînement et de validation restent égales : le modèle ne sur-apprend pas, il est trop simple. Une régression logistique sépare les classes par des frontières linéaires, ce qui plafonne vers 92 % sur MNIST.

## Avec un optimiseur Keras

La mise à jour écrite à la main ci-dessus (`variable.assign_sub(learning_rate * gradient)`) est exactement ce que fait l'optimiseur `keras.optimizers.SGD`. Les optimiseurs de Keras s'utilisent sans modèle Keras, sur de simples `tf.Variable` : `optimizer.apply_gradients(zip(gradients, variables))` remplace la boucle de mise à jour (slide « PMC en TensorFlow de base : un pas d'entraînement »).

*Complétez la boucle de `train_with_optimizer`, qui fait un pas de `optimizer` par batch. La cellule qui suit la solution entraîne un `SimpleNN` avec `keras.optimizers.SGD` et le même learning rate que précédemment, puis un autre avec `keras.optimizers.Adam`.*

**Documentation utile**

- [`keras.optimizers.SGD`](https://keras.io/api/optimizers/sgd/)

In [ ]:
def train_with_optimizer(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Votre code ici

    val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1} : val_acc {val_accuracy:.4f}")

### Conseils supplémentaires

- La perte et les gradients se calculent comme dans `train`, sans régularisation
- `optimizer.apply_gradients(zip(gradients, model.trainable_variables))` : un pas de l'optimiseur sur toutes les variables

Un optimiseur Keras crée ses propres variables (les moyennes d'Adam, par exemple) pour les variables qu'il voit au premier pas : un nouveau modèle demande un nouvel optimiseur.

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_batch, y_batch in dataset_train:
  with tf.GradientTape() as tape:
    loss_value = categorical_crossentropy(..., ...)
  gradients = tape.gradient(..., ...)
  optimizer.apply_gradients(zip(..., ...))
```

### Solution

In [ ]:
def train_with_optimizer(
  model: tf.Module,
  optimizer: keras.optimizers.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      with tf.GradientTape() as tape:
        loss_value = categorical_crossentropy(y_batch, model(X_batch))
      gradients = tape.gradient(loss_value, model.trainable_variables)
      optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1} : val_acc {val_accuracy:.4f}")

### Comparaison de SGD et d'Adam

5 epochs avec chacun : SGD avec le learning rate de la boucle écrite à la main, Adam avec son learning rate usuel, bien plus petit.

In [ ]:
print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, keras.optimizers.SGD(learning_rate=0.5))

print("Adam")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, keras.optimizers.Adam(learning_rate=1e-3))

*Obtenez-vous avec SGD les performances de la boucle écrite à la main ? Que change Adam ?*

Votre réponse ici

#### Solution

Avec SGD et le même learning rate, on retrouve les performances de la boucle écrite à la main (environ 92 % en 5 epochs) : `apply_gradients` applique la même règle de mise à jour. Changer d'algorithme d'optimisation ne demande plus que de changer d'optimiseur. Adam adapte le pas de chaque variable à l'historique de ses gradients : avec un learning rate 500 fois plus petit, il arrive au même résultat. Sur ce modèle simple, il n'apporte rien de plus ; son intérêt est que la valeur 10⁻³ convient à la plupart des réseaux, quand celle de SGD est à chercher pour chacun.

## Compiler le pas d'entraînement avec `tf.function`

Jusqu'ici, TensorFlow exécute chaque opération dès que Python l'appelle (mode *eager*). Décorée par `tf.function`, une fonction est d'abord *tracée* : son code Python s'exécute une fois, et TensorFlow enregistre ses opérations dans un graphe, qu'il exécute ensuite sans repasser par Python (slides « `tf.function` : tracer un graphe » et « `tf.function` : le piège du traçage »).

La cellule suivante définit `train_step`, un pas d'entraînement de `model` par `optimizer` (le corps de votre boucle précédente), et `time_epoch(step)`, qui chronomètre une epoch faite avec la fonction `step`. Elle affiche la durée d'une epoch en mode eager.

*Créez `graph_train_step`, la version compilée de `train_step` par `tf.function`, et chronométrez deux epochs avec elle. Laquelle des deux versions est la plus rapide, et pourquoi ? Que se passe-t-il au premier appel de `graph_train_step` ?*

In [ ]:
model = SimpleNN()
optimizer = keras.optimizers.SGD(learning_rate=0.5)
dataset_train = create_dataset(X_train, y_train)


def train_step(X_batch: tf.Tensor, y_batch: tf.Tensor) -> tf.Tensor:
  with tf.GradientTape() as tape:
    loss_value = categorical_crossentropy(y_batch, model(X_batch))
  gradients = tape.gradient(loss_value, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))
  return loss_value


def time_epoch(step: Callable[[tf.Tensor, tf.Tensor], tf.Tensor]) -> float:
  start = time.perf_counter()
  for X_batch, y_batch in dataset_train:
    step(X_batch, y_batch)
  return time.perf_counter() - start


print(f"Eager : {time_epoch(train_step):.2f} s")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tf.function(train_step)` : la version compilée, à appeler comme `train_step`
- `time_epoch(graph_train_step)` : la durée d'une epoch avec elle, en secondes

```python
graph_train_step = tf.function(...)
print(f"Compilé, epoch 1 : {time_epoch(...):.2f} s")
print(f"Compilé, epoch 2 : {...:.2f} s")
```

### Solution

In [ ]:
graph_train_step = tf.function(train_step)
# Le premier appel trace la fonction
print(f"Compilé, epoch 1 : {time_epoch(graph_train_step):.2f} s")
print(f"Compilé, epoch 2 : {time_epoch(graph_train_step):.2f} s")

La version compilée est 2 à 4 fois plus rapide, selon la machine : le graphe s'exécute sans repasser par Python à chaque opération, et TensorFlow peut l'optimiser (fusion d'opérations). Le gain est d'autant plus grand que les opérations sont petites : ici, le surcoût de Python pèse lourd face à des produits matriciels de 128 exemples.

Le premier appel de `graph_train_step` trace la fonction : son code Python s'exécute une fois pour construire le graphe. Le dernier batch, de 80 exemples (50 000 = 390 $\times$ 128 + 80), a une autre forme et déclenche un second traçage. Ces traçages ne coûtent qu'une fraction de seconde, aussi la première epoch compilée n'est-elle souvent pas plus lente que les suivantes. En revanche, une fonction appelée avec des formes toujours nouvelles serait retracée à chaque appel, et plus lente qu'en mode eager.

## Passage à des réseaux profonds

Pour passer à un perceptron multicouche, on crée une matrice de poids et un vecteur de biais par couche. La fonction suivante les range dans une liste, un dictionnaire `{"W": ..., "b": ...}` par couche :

In [ ]:
def create_mlp_weights(
  n_inputs: int, n_outputs: int, hidden_layer_sizes: tuple[int, ...] = (512,)
) -> list[dict[str, tf.Variable]]:
  layers = []
  # La taille d'entrée d'une couche est la taille de sortie de la précédente
  last_size = n_inputs
  for size in [*hidden_layer_sizes, n_outputs]:
    layers.append(
      {"W": init_weights(last_size, size), "b": tf.Variable(tf.zeros(size))}
    )
    last_size = size
  return layers

La sortie du modèle se calcule maintenant couche par couche : chaque couche prend en entrée le résultat de la précédente.

*Complétez `MLP.__call__`. Chaque couche cachée calcule $\text{ReLU}(XW + b)$ avec `tf.nn.relu` ; la dernière renvoie les logits $XW + b$, sans activation. Les variables de la couche `i` sont `self.layers[i]["W"]` et `self.layers[i]["b"]`.*

In [ ]:
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    pass  # Votre code ici

### Conseils supplémentaires

- `self.layers[:-1]` : les couches cachées ; `self.layers[-1]` : la couche de sortie
- `tf.nn.relu(X @ layer["W"] + layer["b"])` : une couche cachée

Attendu : `MLP()(X_train[:5])` a la forme (5, 10), et `MLP((256, 128))` a 6 variables (`trainable_variables`).

```python
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    for layer in self.layers[:-1]:
      X = tf.nn.relu(...)
    output_layer = self.layers[-1]
    return ...
```

### Solution

In [ ]:
class MLP(tf.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def __call__(self, X: tf.Tensor) -> tf.Tensor:
    # Couches cachées : transformation affine, puis ReLU
    for layer in self.layers[:-1]:
      X = tf.nn.relu(X @ layer["W"] + layer["b"])
    # Couche de sortie : les logits, sans activation
    output_layer = self.layers[-1]
    return X @ output_layer["W"] + output_layer["b"]


print(MLP()(X_train[:5]).shape, len(MLP((256, 128)).trainable_variables))

## Entraînement du réseau profond

Une couche cachée de 512 neurones, 10 epochs, un learning rate de 0,5 :

In [ ]:
mlp = MLP()
_ = train(mlp, epochs=10, learning_rate=0.5)

*Que constatez-vous en comparant ce réseau au réseau sans couche cachée ?*

Votre réponse ici

### Solution

Le réseau à une couche cachée fait nettement mieux que le réseau sans couche cachée : environ 98 % de justesse en validation, contre 92,5 %, soit une erreur divisée par 3. La couche cachée et sa ReLU rendent le modèle non linéaire. La justesse d'entraînement dépasse 99 % et s'écarte de celle de validation : le réseau commence à sur-apprendre, mais sa perte de validation baisse encore.

C'est en partie grâce à l'initialisation des poids, de variance $1 / n_\text{in}$. Pour vous en convaincre, retirez la division par `n_in**0.5` dans `init_weights` et relancez : les logits initiaux ont un écart type de l'ordre de 100, la perte initiale vaut environ 240 au lieu de 2,3, et le réseau n'atteint qu'environ 91 % en validation après 2 epochs, contre 96 %.

## Régularisation

Le PMC précédent sur-apprend peu : sa perte de validation baisse encore à la dixième epoch, car 50 000 exemples suffisent à ses 400 000 poids. Le sur-apprentissage apparaît quand le réseau est grand par rapport aux données : la cellule qui suit la solution entraîne le même PMC sur les 1 000 premiers exemples seulement, 100 epochs, sans puis avec régularisation.

La régularisation L2 ajoute à la perte une pénalité, $\lambda \sum w^2$ : $\lambda$ fois la somme des carrés des poids, qui pousse les poids vers 0. On ne pénalise que les poids `W`, pas les biais : ils ne multiplient pas l'entrée, et les contraindre n'aide pas à généraliser.

*Complétez la fonction interne de `l2`, qui renvoie la pénalité d'un `MLP` : la somme des carrés des matrices `W` de toutes ses couches (`model.layers`), multipliée par `lambda_coefficient`.*

**Documentation utile**

- [`tf.reduce_sum`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_sum)

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    pass  # Votre code ici

  return penalty

### Conseils supplémentaires

- `tf.reduce_sum(layer["W"] ** 2)` : la somme des carrés d'une matrice
- `sum(...)` de Python, ou `tf.add_n([...])`, pour additionner celles de toutes les couches

Attendu : pour un `MLP()` neuf, `l2(1.0)(MLP())` vaut environ 522 (512 pour la première couche, 10 pour la seconde : chacune contribue à peu près son nombre de colonnes, puisque chaque poids a une variance de $1 / n_\text{in}$).

```python
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    weights_norm = sum(tf.reduce_sum(...) for layer in model.layers)
    return ... * weights_norm

  return penalty
```

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> tf.Tensor:
    # Les poids W seulement, pas les biais
    weights_norm = sum(tf.reduce_sum(layer["W"] ** 2) for layer in model.layers)
    return lambda_coefficient * weights_norm

  return penalty


print(float(l2(1.0)(MLP())))

### Sur-apprentissage sur 1 000 exemples

Le même PMC, avec le même learning rate de 0,5, sur les 1 000 premiers exemples d'entraînement, sans régularisation puis avec une pénalité L2 de $\lambda$ = 10⁻³ ; la validation reste faite sur les 10 000 exemples mis de côté.

In [ ]:
X_small, y_small = X_train[:1000], y_train[:1000]

print("Sans régularisation")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
)

print("Avec une pénalité L2")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
  regularization=l2(1e-3),
)

*Comparez les deux entraînements : l'écart entre entraînement et validation, la perte de validation au fil des epochs, la justesse de validation finale.*

Votre réponse ici

#### Solution

Sans régularisation, le réseau apprend par cœur les 1 000 exemples : 100 % de justesse d'entraînement dès la vingtième epoch, une perte d'entraînement qui tend vers 0, et une justesse de validation bloquée vers 89,5 %. La perte de validation passe par un minimum vers la vingtième epoch, puis remonte : le réseau devient de plus en plus sûr de ses réponses, y compris des fausses. C'est le sur-apprentissage.

Avec la pénalité L2, la perte d'entraînement reste au-dessus de 0,02 (les poids ne peuvent plus grandir librement) et la perte de validation ne remonte plus. La justesse de validation gagne peu, quelques dixièmes de point : sur si peu de données, la régularisation limite surtout l'excès de confiance du modèle. Rien ne remplace davantage de données : le même réseau atteint 98 % avec les 50 000 exemples.

## Recherche d'hyper-paramètres

Le learning rate et le nombre de neurones cachés sont des hyper-paramètres : on les choisit en comparant des modèles sur l'ensemble de validation, jamais sur le test. Une recherche aléatoire tire chaque essai au hasard parmi les valeurs possibles.

*Complétez `search_hyperparameters` : à chaque itération, tirez un learning rate dans `learning_rates` et un nombre de neurones cachés dans `ns_hidden_units` avec `random.choice`, entraînez un `MLP` à une couche cachée pendant `epochs` epochs, puis notez ses paramètres et sa justesse de validation finale. La fonction renvoie la liste des paramètres essayés et celle des justesses. Lequel des deux hyper-paramètres compte le plus ?*

**Documentation utile**

- [`random.choice`](https://docs.python.org/fr/3/library/random.html#random.choice)

In [ ]:
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  pass  # Votre code ici


search_hyperparameters()

### Conseils supplémentaires

- `random.choice(learning_rates)` : une valeur tirée au hasard
- `MLP(hidden_layer_sizes=(n_hidden_units,))` : une couche cachée de `n_hidden_units` neurones
- `train(mlp, learning_rate=..., epochs=epochs, evaluate_every=epochs)` renvoie quatre listes : la deuxième contient les justesses de validation, la dernière est celle de fin d'entraînement (`evaluate_every=epochs` : une seule évaluation, à la fin)

```python
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  params, val_accs = [], []
  for _ in range(n_iters):
    learning_rate = random.choice(...)
    n_hidden_units = ...
    mlp = MLP(hidden_layer_sizes=(...,))
    _, val_accuracies, _, _ = train(
      mlp, learning_rate=..., epochs=epochs, evaluate_every=epochs
    )
    params.append({"learning_rate": ..., "n_hidden_units": ...})
    val_accs.append(...)
  return params, val_accs
```

### Solution

In [ ]:
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  params, val_accs = [], []
  for _ in range(n_iters):
    learning_rate = random.choice(learning_rates)
    n_hidden_units = random.choice(ns_hidden_units)
    mlp = MLP(hidden_layer_sizes=(n_hidden_units,))
    _, val_accuracies, _, _ = train(
      mlp, learning_rate=learning_rate, epochs=epochs, evaluate_every=epochs
    )
    params.append({"learning_rate": learning_rate, "n_hidden_units": n_hidden_units})
    val_accs.append(val_accuracies[-1])
  return params, val_accs


params, val_accs = search_hyperparameters()
for p, val_acc in sorted(zip(params, val_accs), key=lambda pair: -pair[1]):
  print(
    f"lr {p['learning_rate']}, {p['n_hidden_units']} neurones : val_acc {val_acc:.4f}"
  )

Le learning rate compte bien plus que le nombre de neurones : en 3 epochs, les essais à 0,01 plafonnent vers 89 %, ceux à 0,1 dépassent 94 % et ceux à 0,5 approchent 97 %, quelle que soit la taille de la couche cachée. Avec 5 essais, une recherche aléatoire peut manquer la meilleure valeur : le nombre d'essais est un compromis avec le temps de calcul. Et 3 epochs favorisent ce qui apprend vite, pas forcément ce qui finit le mieux.

## Évaluation finale sur l'ensemble de test

L'ensemble de test n'a servi à rien jusqu'ici : ni à l'entraînement, ni au choix des hyper-paramètres. C'est ce qui rend son évaluation fiable.

*Entraînez `mlp`, un MLP aux meilleurs hyper-paramètres trouvés, pendant 10 epochs, puis calculez sa justesse sur `X_test`. Est-elle proche de la justesse de validation ?*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `params[int(np.argmax(val_accs))]` : les paramètres de l'essai à la meilleure justesse de validation
- `MLP(hidden_layer_sizes=(n,))` et `train(mlp, learning_rate=..., epochs=10)`, avec ces valeurs
- `accuracy(y_test, mlp(X_test))` : la justesse de test

```python
best = params[...]
mlp = MLP(hidden_layer_sizes=(best[...],))
_ = train(mlp, learning_rate=..., epochs=10)
print(f"Justesse sur l'ensemble de test : {accuracy(..., ...):.4f}")
```

### Solution

In [ ]:
# Les hyper-paramètres de la meilleure justesse de validation
best = params[int(np.argmax(val_accs))]
print(best)
mlp = MLP(hidden_layer_sizes=(best["n_hidden_units"],))
_ = train(mlp, learning_rate=best["learning_rate"], epochs=10)
print(f"Justesse sur l'ensemble de validation : {accuracy(y_val, mlp(X_val)):.4f}")
print(f"Justesse sur l'ensemble de test : {accuracy(y_test, mlp(X_test)):.4f}")

La justesse de test est proche de la justesse de validation, souvent quelques dixièmes de point en dessous (par exemple 96,7 % contre 97,1 %). La validation a servi à choisir les hyper-paramètres : sa justesse est légèrement optimiste. L'ensemble de test, qui n'a servi à aucun choix, donne une estimation fiable des performances du modèle sur de nouvelles données.

## Sauvegarder et restaurer le modèle

Un entraînement coûte cher : on sauvegarde le modèle entraîné pour le réutiliser sans le réentraîner. `tf.train.Checkpoint` enregistre les valeurs des variables d'un objet, retrouvées par leurs noms d'attributs (slide « Sauvegarder et restaurer : `tf.train.Checkpoint` »).

*Sauvegardez `mlp` dans `checkpoints/mlp`. Créez `restored`, un nouveau `MLP` de même architecture, et affichez sa justesse de test avant et après y avoir restauré le checkpoint.*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `tf.train.Checkpoint(model=mlp).save("checkpoints/mlp")` : la sauvegarde ; elle renvoie le chemin du checkpoint, `checkpoints/mlp-1` (le numéro augmente à chaque sauvegarde)
- `tf.train.Checkpoint(model=restored).restore(chemin)` : la restauration, sous le même nom d'argument, `model`
- `restored` doit avoir la même architecture : mêmes `hidden_layer_sizes` que `mlp`

```python
path = tf.train.Checkpoint(model=...).save("checkpoints/mlp")
restored = MLP(hidden_layer_sizes=(...,))
print(accuracy(y_test, restored(X_test)))
tf.train.Checkpoint(model=...).restore(...)
print(accuracy(..., ...))
```

### Solution

In [ ]:
path = tf.train.Checkpoint(model=mlp).save("checkpoints/mlp")
print(path)

# Nouveaux poids aléatoires, même architecture que mlp
restored = MLP(hidden_layer_sizes=(best["n_hidden_units"],))
print(f"Avant restauration : {accuracy(y_test, restored(X_test)):.4f}")
tf.train.Checkpoint(model=restored).restore(path)
print(f"Après restauration : {accuracy(y_test, restored(X_test)):.4f}")
print(f"Modèle sauvegardé : {accuracy(y_test, mlp(X_test)):.4f}")

Avant restauration, `restored` répond presque au hasard (de l'ordre de 10 %, ses poids sont aléatoires). Après, il a exactement la justesse de `mlp` : le checkpoint a rendu à chaque variable sa valeur, retrouvée par son chemin d'attributs (`model`, puis `layers`, puis l'indice et la clé). D'où l'importance d'une architecture identique : un `MLP` d'une autre taille aurait des variables de formes différentes, et la restauration échouerait.

Un checkpoint ne contient que des valeurs, pas le code du modèle : pour le restaurer, il faut la classe `MLP`. Pour déployer un modèle sans son code Python, TensorFlow a un autre format, le SavedModel (`tf.saved_model.save`).